# Chapter 3

## Summarizing a document bigger than the LLM’s context window

In [3]:
# Install required packages
!pip install langchain-google-genai google-generativeai

In [4]:
!pip install tiktoken notebook langchain langchain-google-genai google-generativeai langchain-community langchain-core langchain-text-splitters wikipedia docx2txt pypdf

In [5]:
with open("./Moby-Dick-Short.txt", 'r', encoding='utf-8') as f:
    moby_dick_book = f.read()

In [6]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_text_splitters import TokenTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel
import os

In [7]:
from google.colab import userdata
GOOGLE_API_KEY = userdata.get('APIKeyForLLMs')

In [8]:
llm = ChatGoogleGenerativeAI(google_api_key=GOOGLE_API_KEY, model="gemini-2.5-flash")

In [9]:
print(llm.model)

gemini-2.5-flash


In [10]:
# Map
summarize_chunk_prompt_template = """
Write a concise summary of the following text, and include the main details.
Text: {chunk}
"""

summarize_chunk_prompt = PromptTemplate.from_template(summarize_chunk_prompt_template)
summarize_chunk_chain = summarize_chunk_prompt | llm

summarize_map_chain = (
    RunnableParallel (
        {
            'summary': summarize_chunk_chain | StrOutputParser()
        }
    )
)

In [11]:
# Reduce
summarize_summaries_prompt_template = """
Write a coincise summary of the following text, which joins several summaries, and include the main details.
Text: {summaries}
"""

summarize_summaries_prompt = PromptTemplate.from_template(summarize_summaries_prompt_template)
summarize_reduce_chain = (
    RunnableLambda(lambda x:
        {
            'summaries': '\n'.join([i['summary'] for i in x]),
        })
    | summarize_summaries_prompt
    | llm
    | StrOutputParser()
)

In [12]:
# Split
text_chunks_chain = (
    RunnableLambda(lambda x:
        [
            {
                'chunk': text_chunk,
            }
            for text_chunk in
               TokenTextSplitter(chunk_size=3000, chunk_overlap=100).split_text(x)
        ]
    )
)

In [13]:
map_reduce_chain = (
   text_chunks_chain
   | summarize_map_chain.map()
   | summarize_reduce_chain
)

In [14]:
summary = map_reduce_chain.invoke(moby_dick_book)

In [15]:
print(summary)

The narrator, Ishmael, introduces himself by explaining his recurrent habit of going to sea to escape melancholy and suicidal thoughts, a universal human urge he links to water. He always sails as a simple sailor, embracing the role and avoiding responsibility. His current intention is a whaling voyage, driven by an overwhelming curiosity about whales, a love for remote and dangerous seas, and a desire for adventure, which he perceives as both fated and a personal choice.

Ishmael recounts his journey from Manhattan to New Bedford, arriving on a cold December Saturday night. Having missed the packet to Nantucket—his preferred port due to its whaling history—he is forced to wait until Monday. With limited funds, he searches for cheap lodging in the unfamiliar town, eventually settling on the dilapidated and ominously named "The Spouter Inn," run by Peter Coffin, as his best available option.


## Summarizing across documents

In [16]:
from langchain_community.document_loaders import WikipediaLoader

wikipedia_loader = WikipediaLoader(query="Paestum", load_max_docs=2)
wikipedia_docs = wikipedia_loader.load()

In [17]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import TextLoader

word_loader = Docx2txtLoader("Paestum-Britannica.docx")
word_docs = word_loader.load()

pdf_loader = PyPDFLoader("PaestumRevisited.pdf")
pdf_docs = pdf_loader.load()

txt_loader = TextLoader("Paestum-Encyclopedia.txt")
txt_docs = txt_loader.load()

In [18]:
all_docs = wikipedia_docs + word_docs + pdf_docs + txt_docs

In [19]:
doc_summary_template = """Write a concise summary of the following text:
{text}
DOC SUMMARY:"""
doc_summary_prompt = PromptTemplate.from_template(doc_summary_template)

doc_summary_chain = doc_summary_prompt | llm

In [20]:
refine_summary_template = """
Your must produce a final summary from the current refined summary
which has been generated so far and from the content of an additional document.
This is the current refined summary generated so far: {current_refined_summary}
This is the content of the additional document: {text}
Only use the content of the additional document if it is useful,
otherwise return the current full summary as it is."""

refine_summary_prompt = PromptTemplate.from_template(refine_summary_template)

refine_chain = refine_summary_prompt | llm | StrOutputParser()

In [21]:
def refine_summary(docs):

    intermediate_steps = []
    current_refined_summary = ''
    for doc in docs:
        intermediate_step = \
           {"current_refined_summary": current_refined_summary,
            "text": doc.page_content}
        intermediate_steps.append(intermediate_step)

        current_refined_summary = refine_chain.invoke(intermediate_step)

    return {"final_summary": current_refined_summary,
            "intermediate_steps": intermediate_steps}

In [22]:
full_summary = refine_summary(all_docs)
print(full_summary['final_summary'])

Paestum, originally a major ancient Greek city named Poseidonia, was founded around 600 BCE by settlers from Sybaris, a mother colony whose founder Strabo identified as Is of Helice, located along the Gulf of Taranto, on the Tyrrhenian Sea coast in Magna Graecia. It had become a flourishing town by 540 BCE. Its remarkably well-preserved ruins are today located within the modern comune of Capaccio Paestum, in the province of Salerno, Campania, Italy, situated in northern Cilento, 22 miles (35 km) southeast of modern Salerno and 5 miles (8 km) south of the Sele (ancient Silarus) River.

Poseidonia flourished as an autonomous Greek polis for some 200 years, becoming a significant commercial and cultural center, even filling the gap after its mother colony, Sybaris, fell in 510 BCE. Sometime before 400 BCE, the city came under the control of the Lucanians, a south Italian Samnite people, who were expanding their territory. While earlier scholarship often described this as a period of force